In [1]:
!pip install zeep


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 14.9 MB/s  0:00:00eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [zeep]3/4 [zeep]


In [30]:
from uuid import uuid4
import requests
from zeep import Client
from zeep.transports import Transport

WSDL_URL = "https://www.easylaw.go.kr/OPENAPI/soap/LifeLawInfoService?wsdl"
SERVICE_KEY = "efa6fa34f0bc4c7b2ea664897d6772397089a2d60e7b2fb1e5657dcd41f75cf1"

session = requests.Session()
transport = Transport(session=session, timeout=30)
client = Client(wsdl=WSDL_URL, transport=transport)

# 1) Body (요청 메시지) - 명세대로!
req = {
    # 명세에 ServiceKey가 "필수(1)"로 잡혀있으면 넣어줘야 함
    "ServiceKey": SERVICE_KEY,

    # 아래 4개가 이 오퍼레이션의 핵심 필수값
    "csmSeq": 521,     # 생활분야일련번호 (예시)
    "ccfNo": 3,        # 관심분야일련번호 (예시)
    "cciNo": 1,        # 관심항목일련번호 (예시)
    "cnpClsNo": 1,     # 관심규정일련번호 (※ 여기 '실제값'이 중요)
}

# 2) SOAP Header (ComMsgHeader)
ComMsgHeaderType = client.get_type("ns1:ComMsgHeader")
com_header = ComMsgHeaderType(
    RequestMsgID=str(uuid4()),
    ServiceKey=SERVICE_KEY,
)

# 3) Call
result = client.service.getLifeLawsInterpretList(
    req,
    _soapheaders={"ComMsgHeader": com_header}
)

print(result)


{
    'header': {
        'ComMsgHeader': {
            'RequestMsgID': '85dcd766-d2df-436f-9d7a-60ec93425d09',
            'ServiceKey': 'efa6fa34f0bc4c7b2ea664897d6772397089a2d60e7b2fb1e5657dcd41f75cf1',
            'RequestTime': None,
            'CallBackURI': None
        }
    },
    'body': {
        'LifeLawsInterpretListItem': {
            'errMsg': None,
            'returnCode': None,
            'totcnt': None,
            'lifeLawsInterpretExpcListCount': 2,
            'LifeLawsInterpretExpcListItems': {
                'LifeLawsInterpretExpcListItem': [
                    {
                        'ccfNo': '3',
                        'cciNm': '사업면허 등 준비절차',
                        'cciNo': '1',
                        'cnpClsNm': '신청절차와 면허의 요건',
                        'cnpClsNo': '1',
                        'csmSeq': '521',
                        'expcNo': '3',
                        'inq': '사업용으로 등록된 자동차에 대해 저당권등록이 되어 있을 경우 사업면허권까지 포함한 저당권등록인지 아니면 등록된 해당 자동차 자체에

In [3]:
import json
import asyncio
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse, parse_qs

from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError
from bs4 import BeautifulSoup


# =========================
# Config (피해구제 사례 115)
# =========================
BASE = "https://www.consumer.go.kr"

LIST_URL_TMPL = (
    "https://www.consumer.go.kr/user/ftc/consumer/dmgerlifcase/115/selectDmgeRlifCaseList.do"
    "?page={page}&row=25&searchType=&searchCnd=&searchWrd="
)

OUT_JSONL = "dmge_rlif_cases_full.jsonl"

HEADLESS = True
TIMEOUT_MS = 30_000
MAX_RETRIES = 3
RETRY_BACKOFF_SEC = 1.5
CHECKPOINT_EVERY = 1

LIST_LINK_SELECTOR = "td.title a"          # ✅ 네 HTML과 동일
DETAIL_TABLE_SELECTOR = "table.tbl.row.data"  # ✅ 상세 표 (네 HTML과 동일)


# =========================
# Helpers
# =========================
def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def extract_dmge_sn(url: str) -> str | None:
    qs = parse_qs(urlparse(url).query)
    v = qs.get("dmgeRlifCaseSn")
    return v[0] if v else None

def make_doc_id(url: str) -> str:
    sn = extract_dmge_sn(url)
    if sn:
        return str(sn)
    # fallback
    p = urlparse(url)
    normalized = f"{p.scheme}://{p.netloc}{p.path}?{p.query}"
    return f"url:{normalized}"

def safe_text(el) -> str:
    if not el:
        return ""
    return el.get_text("\n", strip=True)

def parse_detail_html(html: str, url: str) -> dict:
    """
    네가 붙여준 상세 HTML 구조:
    <table class="tbl row data">
      <tr><th>제목</th><td>...</td></tr>
      <tr><th>분류</th><td>...</td></tr>
      <tr><th>출처</th><td><a ...>...</a></td></tr>
      <tr><th>질문</th><td><div class="bbs_view_content">...</div></td></tr>
      <tr><th>답변</th><td><div class="bbs_view_content">...</div></td></tr>
    """
    soup = BeautifulSoup(html, "html.parser")

    def get_row(label: str) -> str:
        # th 텍스트가 약간 바뀌어도 포함 매칭으로 최대한 견고하게
        for th in soup.find_all("th"):
            if label in th.get_text(" ", strip=True):
                td = th.find_next_sibling("td")
                if not td:
                    return ""
                div = td.select_one("div.bbs_view_content")
                if div:
                    return safe_text(div)
                return safe_text(td)
        return ""

    title = get_row("제목")
    category = get_row("분류")
    source = get_row("출처")
    question = get_row("질문")
    answer = get_row("답변")

    doc_id = make_doc_id(url)

    # ✅ RAG/임베딩용 content 한 덩어리
    parts = []
    if title:
        parts.append(f"제목: {title}")
    if category:
        parts.append(f"분류: {category}")
    if source:
        parts.append(f"출처: {source}")
    if question:
        parts.append(f"질문:\n{question}")
    if answer:
        parts.append(f"답변:\n{answer}")
    content = "\n\n".join(parts).strip()

    return {
        "id": doc_id,
        "url": url,
        "title": title,
        "category": category,
        "source": source,
        "question": question,
        "answer": answer,
        "content": content,
        "collected_at": now_iso(),
        "metadata": {
            "site": "consumer.go.kr",
            "doc_type": "dmge_rlif_case",  # 피해구제 사례
            "dmgeRlifCaseSn": extract_dmge_sn(url),
        },
    }

def load_seen_ids(out_jsonl: str) -> set[str]:
    seen = set()
    try:
        with open(out_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    _id = obj.get("id")
                    if _id:
                        seen.add(str(_id))
                except Exception:
                    pass
    except FileNotFoundError:
        pass
    return seen

def append_jsonl(fp, obj: dict):
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")
    fp.flush()


# =========================
# Speed: block heavy assets
# =========================
async def block_heavy_assets(route):
    rtype = route.request.resource_type
    if rtype in ("image", "font", "media"):
        await route.abort()
    else:
        await route.continue_()


# =========================
# Main
# =========================
async def main(start_page=1, end_page=100):
    seen = load_seen_ids(OUT_JSONL)
    print("seen already:", len(seen))

    total_saved = 0
    total_skipped = 0
    total_failed = 0

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)

        context = await browser.new_context()
        context.set_default_timeout(TIMEOUT_MS)
        await context.route("**/*", block_heavy_assets)

        list_page = await context.new_page()
        detail_page = await context.new_page()

        with open(OUT_JSONL, "a", encoding="utf-8") as out_fp:
            for pg in range(start_page, end_page + 1):
                list_url = LIST_URL_TMPL.format(page=pg)

                try:
                    await list_page.goto(list_url, wait_until="domcontentloaded")
                    # ✅ 목록 링크가 뜰 때까지 기다림
                    await list_page.wait_for_selector(LIST_LINK_SELECTOR)
                except PWTimeoutError:
                    # 페이지가 실제로 비어있는지 확인용: tr 개수도 같이 찍어주면 디버깅 쉬움
                    tr_cnt = await list_page.locator("tbody tr").count()
                    print(f"[list][timeout] page={pg} tr={tr_cnt} url={list_url}")
                    continue

                hrefs = await list_page.locator(LIST_LINK_SELECTOR).evaluate_all(
                    "els => els.map(a => a.getAttribute('href'))"
                )
                links = [urljoin(BASE, h) for h in hrefs if h]
                print(f"[list] page={pg} links={len(links)}")

                for link in links:
                    doc_id = make_doc_id(link)
                    if doc_id in seen:
                        total_skipped += 1
                        continue

                    last_err = None
                    ok = False

                    for attempt in range(1, MAX_RETRIES + 1):
                        try:
                            await detail_page.goto(link, wait_until="domcontentloaded")
                            # ✅ 상세 표가 뜨는지 확인
                            await detail_page.wait_for_selector(DETAIL_TABLE_SELECTOR)

                            html = await detail_page.content()
                            item = parse_detail_html(html, link)

                            # 최소 검증: 질문/답변/제목 중 하나는 있어야 저장
                            if not item["title"] and not item["question"] and not item["answer"]:
                                raise RuntimeError("empty parsed fields")

                            item["list_page"] = pg
                            append_jsonl(out_fp, item)

                            seen.add(item["id"])
                            total_saved += 1
                            ok = True
                            break

                        except Exception as e:
                            last_err = e
                            if attempt < MAX_RETRIES:
                                await asyncio.sleep(RETRY_BACKOFF_SEC * attempt)
                            else:
                                total_failed += 1
                                print(f"  [fail] page={pg} id={doc_id} err={repr(last_err)}")

                    if not ok:
                        pass

                if pg % CHECKPOINT_EVERY == 0:
                    print(
                        f"✅ checkpoint page {pg} | saved={total_saved} skipped={total_skipped} failed={total_failed}"
                    )

        await browser.close()

    print("\n==== DONE ====")
    print(f"pages: {start_page}~{end_page}")
    print(f"saved: {total_saved}")
    print(f"skipped(seen): {total_skipped}")
    print(f"failed: {total_failed}")


# 노트북에서 실행
# await main(start_page=1, end_page=50)
await main(start_page=1, end_page=3)


seen already: 0
[list] page=1 links=25
✅ checkpoint page 1 | saved=25 skipped=0 failed=0
[list] page=2 links=25
✅ checkpoint page 2 | saved=50 skipped=0 failed=0
[list] page=3 links=25
✅ checkpoint page 3 | saved=75 skipped=0 failed=0

==== DONE ====
pages: 1~3
saved: 75
skipped(seen): 0
failed: 0


In [31]:
from pathlib import Path
import json
from collections import Counter, defaultdict

DATA_ROOT = Path("../../data")  # 현재: ism/scripts/preprocess 기준

datasets = ["cnslt", "dmge_rlif", "kca_00000006", "trubl_mdat"]

def unique_sources_in_jsonl(jsonl_path: Path):
    counter = Counter()
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            obj = json.loads(s)
            counter[obj.get("source")] += 1
    return counter

results = defaultdict(dict)

for ds in datasets:
    ds_dir = DATA_ROOT / ds

    # processed 폴더 우선, 없으면 ds_dir 전체에서 탐색
    search_root = (ds_dir / "processed") if (ds_dir / "processed").exists() else ds_dir

    # clean 우선 탐색
    files = list(search_root.rglob("*.processed.clean.jsonl"))

    # clean이 없으면 processed.jsonl도 fallback
    if not files:
        files = list(search_root.rglob("*.processed.jsonl"))

    print(f"\n=== {ds} ===")
    if not files:
        print("  [NOT FOUND] processed(.clean).jsonl 파일을 못 찾음")
        continue

    # 여러 개면 전부 돌림
    for fp in sorted(files):
        counter = unique_sources_in_jsonl(fp)
        results[ds][str(fp)] = counter

        print(f"  file: {fp}")
        print(f"  sources: {list(counter.keys())}")
        print(f"  counts : {dict(counter)}")



=== cnslt ===
  file: ../../data/cnslt/processed/cnslt_cases_114_full.processed.clean.jsonl
  sources: ['1372 소비자 상담센터', '콘텐츠분쟁조정위원회']
  counts : {'1372 소비자 상담센터': 11316, '콘텐츠분쟁조정위원회': 26}
  file: ../../data/cnslt/processed/cnslt_cases_full.processed.clean.jsonl
  sources: ['1372 소비자 상담센터', '콘텐츠분쟁조정위원회']
  counts : {'1372 소비자 상담센터': 11314, '콘텐츠분쟁조정위원회': 26}

=== dmge_rlif ===
  file: ../../data/dmge_rlif/processed/dmge_rlif_cases_115_full.processed.clean.jsonl
  sources: ['한국소비자원', '콘텐츠분쟁조정위원회', '대한법률구조공단', '한국석유관리원', '전자거래분쟁조정위원회']
  counts : {'한국소비자원': 1419, '콘텐츠분쟁조정위원회': 75, '대한법률구조공단': 1, '한국석유관리원': 1, '전자거래분쟁조정위원회': 1}
  file: ../../data/dmge_rlif/processed/dmge_rlif_cases_full.processed.clean.jsonl
  sources: ['콘텐츠분쟁조정위원회', '한국소비자원', '대한법률구조공단', '한국석유관리원', '전자거래분쟁조정위원회']
  counts : {'콘텐츠분쟁조정위원회': 75, '한국소비자원': 1419, '대한법률구조공단': 1, '한국석유관리원': 1, '전자거래분쟁조정위원회': 1}

=== kca_00000006 ===
  file: ../../data/kca_00000006/processed/kca_00000006_full.processed.clean.jsonl
  sources: ['보

In [29]:
import json

FILE_PATH = "../../data/cnslt/processed/cnslt_cases_114_full.processed.clean.jsonl"


sources = set()

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        s = line.strip()
        if not s:
            continue
        obj = json.loads(s)
        if "source" in obj:
            sources.add(obj["source"])

sources


{'1372 소비자 상담센터', '콘텐츠분쟁조정위원회'}

In [30]:
import json

FILE_PATH = "../../data/cnslt/processed/cnslt_cases_114_full.processed.clean.jsonl"


rows_1372 = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        if obj.get("source") == "1372 소비자 상담센터":
            rows_1372.append(obj)

len(rows_1372)


11316

In [26]:
rows_1372[1]


{'doc_id': 'consumer.go.kr:consumer_mediation_case:13361',
 'doc_type': 'mediation_case',
 'title': '대여 유아드레스 훼손에 따른 과다 배상비 및 보증금 환급 요구',
 'category_path': ['의류세탁'],
 'source': '1372 소비자 상담센터',
 'collected_at': '2025-12-29T03:42:40.227833+00:00',
 'text_for_embedding': '[문서유형] mediation_case\n[제목] 대여 유아드레스 훼손에 따른 과다 배상비 및 보증금 환급 요구\n[출처] 1372 소비자 상담센터\n[분류] 의류세탁\n\n[본문]\n[문서유형] 분쟁조정 사례\n\n[제목] 대여 유아드레스 훼손에 따른 과다 배상비 및 보증금 환급 요구\n\n[출처] 1372 소비자 상담센터\n\n[분류] 의류세탁\n\n\n[사건개요]\n신청인은 2018. 8. 7. 피신청인에게서 유아드레스 2벌(사용예정일 : 2018. 9. 6., 9. 9.) 및 헤어 액세서리 세트를 대여하고 270,000원(대여료 170,000원, 보증금 100,000원)을 지급하였는데, 위 드레스를 착용 중 1벌(언니 드레스)이 찢어져서 같은 해 9. 10. 피신청인에게 위 사실을 알리고 반환하자 피신청인은 드레스 훼손이 심해 수선이 어렵다며 드레스 구입비 260,000원의 배상을 요구하여 신청인이 이를 지급하였다. 이후 신청인이 보증금 환급을 요구하자, 피신청인은 계약 당시 약정한 후기 미작성, 함께 대여한 액세서리 세트 중 일부가 반환되지 않았다며 거부하였다.\n[당사자 주장]\n신청인은 현재 상황에서 광고 목적으로 이용할 후기 사진을 제공할 수 없고, 이 사건 훼손된 드레스가 중고임에도 정가를 모두 배상한 것은 부당하다며 잔존가의 차액 환급 및 드레스의 반환을 요구하고, 대여한 액세서리 세트도 드레스와 함께 반환하였다며 보증금 전액의 환급을 요구한다.\n이에 대하여 피신청